# FreshCart ELT on Google Colab: APIs → Snowflake → SQL

This notebook runs the full pipeline from Colab:

1. **Extract**: Python calls the APIs and saves the raw JSON.
2. **Load**: the raw files are uploaded to Snowflake's **bronze** tables (`PUT` + `COPY INTO`).
3. **Transform**: SQL inside Snowflake cleans, deduplicates and enriches into **silver**, then publishes **gold**.

**Before you start**, add your Snowflake login in Colab's **Secrets** panel (the 🔑 key icon in the left sidebar). Switch on *Notebook access* for each one.

| Secret name | Example | Needed? |
|---|---|---|
| `SNOWFLAKE_ACCOUNT` | `myorg-myaccount` (Snowsight → your name → Account → Copy account identifier) | yes |
| `SNOWFLAKE_USER` | `CHANDRA` | yes |
| `SNOWFLAKE_PASSWORD` | your password, **or** a programmatic access token | one of these two |
| `SNOWFLAKE_PRIVATE_KEY` | the full text of an `rsa_key.p8` key-pair file | one of these two |
| `SNOWFLAKE_ROLE` | `SYSADMIN` (default) | optional |

Colab can't open a browser login window, so SSO (`externalbrowser`) does not work here. If your account requires MFA and blocks plain password logins, use a key pair or a programmatic access token instead.

Run the cells from top to bottom.

## 1. Install libraries

In [ ]:
!pip install -q snowflake-connector-python requests python-dotenv

## 2. Create the project files
Each cell below writes one file into `/content/freshcart_elt`. Just run them; you can read them later.

In [ ]:
!mkdir -p /content/freshcart_elt/sql

In [ ]:
%%writefile /content/freshcart_elt/requirements.txt
snowflake-connector-python>=3.6
requests>=2.31
python-dotenv>=1.0

In [ ]:
%%writefile /content/freshcart_elt/mock_data.py
"""Mock API responses: the 8 FreshCart orders from the course, split by the channel
they came from, plus CRM, FX and pincode responses. Used with --mock so the whole
pipeline runs end to end without any real API.

Note the two order APIs use DIFFERENT JSON shapes. That is normal in real life,
and it is why bronze stores raw JSON and silver unifies it with SQL.
"""

# Web export: flat JSON (rows 1, 3, 4, 5, 6, 8 of the course table)
WEB_ORDERS = [
    {"order_id": "ORD-1001", "cust_email": " Priya.R@Gmail.com ", "cust_name": "priya r",
     "order_date": "2026-09-01", "amount": "1,250.00", "currency": "INR", "pincode": "600041",
     "updated_at": "2026-09-01 10:02:00"},
    {"order_id": "ORD-1003", "cust_email": "meera@outlook.com", "cust_name": "MEERA S",
     "order_date": "2026-09-02", "amount": None, "currency": "INR", "pincode": "400050",
     "updated_at": "2026-09-02 09:10:00"},
    {"order_id": "ORD-1002", "cust_email": "Arjun.K@Yahoo.in ", "cust_name": "Arjun K",
     "order_date": "2026-09-01", "amount": "42.50", "currency": "USD", "pincode": "560034",
     "updated_at": "2026-09-01 11:15:00"},
    {"order_id": "ORD-1004", "cust_email": "rahul.m@gmail.com", "cust_name": "Rahul M",
     "order_date": "Sep 3, 2026", "amount": "-500", "currency": "INR", "pincode": "110001",
     "updated_at": "2026-09-03 12:00:00"},
    {"order_id": "ORD-1005", "cust_email": "priya.r@gmail.com", "cust_name": "Priya R.",
     "order_date": "2026-09-03", "amount": "₹ 899", "currency": "inr", "pincode": "600041",
     "updated_at": "2026-09-03 16:45:00"},
    {"order_id": "ORD-1006", "cust_email": "vikram@@mail", "cust_name": "Vikram",
     "order_date": "2026-09-04", "amount": 30, "currency": "EUR", "pincode": "99999",
     "updated_at": "2026-09-04 08:30:00"},
]

# Mobile API: nested JSON, camelCase, day-first dates (rows 2 and 7)
MOBILE_ORDERS = [
    {"orderId": "ORD-1002", "customer": {"email": "arjun.k@yahoo.in", "name": "Arjun K"},
     "orderDate": "01/09/2026", "total": {"value": "42.50", "currency": "usd"},
     "deliveryPin": "560034", "updatedAt": "2026-09-01T11:15:00"},
    {"orderId": "ORD-1003", "customer": {"email": "Meera@Outlook.com", "name": "Meera S"},
     "orderDate": "02/09/2026", "total": {"value": "2,100.00", "currency": "INR"},
     "deliveryPin": "400050", "updatedAt": "2026-09-02T18:40:00"},
]

# CRM: a log of tier changes. The SQL turns this into an SCD Type 2 table.
CRM_TIER_EVENTS = [
    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Silver", "effective_from": "2023-04-10"},
    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Gold", "effective_from": "2026-09-03"},
    {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in", "tier": "Silver", "effective_from": "2024-01-15"},
    {"cust_id": "CUST-03", "email": "meera@outlook.com", "tier": "Platinum", "effective_from": "2022-11-01"},
]

# Same shape as https://api.frankfurter.app/2026-09-01..2026-09-03?from=USD&to=INR
FX_RESPONSES = [
    {"amount": 1.0, "base": "USD", "start_date": "2026-09-01", "end_date": "2026-09-03",
     "rates": {"2026-09-01": {"INR": 83.20}, "2026-09-02": {"INR": 83.35}, "2026-09-03": {"INR": 83.10}}},
]

# Same shape as https://api.postalpincode.in/pincode/600041
_PIN = {"600041": ("Chennai", "Tamil Nadu"), "560034": ("Bengaluru", "Karnataka"),
        "400050": ("Mumbai", "Maharashtra")}


def pincode_response(pin):
    if pin not in _PIN:
        return [{"Message": "No records found", "Status": "Error", "PostOffice": None}]
    district, state = _PIN[pin]
    return [{"Message": "Number of pincode(s) found:1", "Status": "Success",
             "PostOffice": [{"Name": district, "District": district, "State": state,
                             "Country": "India", "Pincode": pin}]}]

In [ ]:
%%writefile /content/freshcart_elt/extract.py
"""EXTRACT: call each API and save the responses unchanged as gzipped newline-delimited
JSON files in the local landing folder.

No cleaning happens here (that is the point of ELT). Each record is wrapped in a
small envelope so we always know where and when it came from:

    {"source": "web", "run_id": "...", "extracted_at": "...", "data": {<original record>}}
"""
import gzip
import json
import logging
import os
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import mock_data

log = logging.getLogger("extract")


# ---------------------------------------------------------------- HTTP helpers
def http_session():
    """A requests session that retries transient failures with exponential backoff
    (1 s, 2 s, 4 s, ...) and honours the server's Retry-After header on 429."""
    retry = Retry(total=5, backoff_factor=1.0,
                  status_forcelist=(429, 500, 502, 503, 504),
                  allowed_methods=("GET",), respect_retry_after_header=True)
    s = requests.Session()
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.mount("http://", HTTPAdapter(max_retries=retry))
    s.headers["User-Agent"] = "freshcart-elt/1.0"
    return s


def _auth(token_env):
    token = os.getenv(token_env)
    return {"Authorization": f"Bearer {token}"} if token else {}


def paginate(session, url, params=None, headers=None, items_key="data", next_key="next",
             timeout=30, max_pages=10_000):
    """Yield records from an API that returns {"data": [...], "next": "<url or null>"}.
    Change items_key / next_key to match your API (e.g. "results" / "next_page_url").
    If your API uses page numbers instead, loop on params["page"] until a page is empty."""
    page_url, page_params = url, dict(params or {})
    for _ in range(max_pages):
        r = session.get(page_url, params=page_params, headers=headers, timeout=timeout)
        r.raise_for_status()
        body = r.json()
        items = body.get(items_key, []) if isinstance(body, dict) else body
        yield from items
        nxt = body.get(next_key) if isinstance(body, dict) else None
        if not nxt:
            return
        page_url, page_params = nxt, None      # the next link already carries the query
    raise RuntimeError(f"{url}: more than {max_pages} pages, stopping")


# ---------------------------------------------------------------- landing files
def write_landing(source, records, run_id, run_date):
    """Write records as one .json.gz file per source per run. Returns the path, or None."""
    records = list(records)
    if not records:
        log.info("%s: 0 records, nothing written", source)
        return None
    folder = Path(os.getenv("LANDING_DIR", "landing")) / source / run_date.isoformat()
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{source}__{run_id}.json.gz"
    now = datetime.now(timezone.utc).isoformat()
    with gzip.open(path, "wt", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps({"source": source, "run_id": run_id, "extracted_at": now,
                                "data": rec}, ensure_ascii=False) + "\n")
    log.info("%s: %d records -> %s", source, len(records), path)
    return path


# ---------------------------------------------------------------- one function per API
def web_orders(session, run_date, mock=False):
    if mock:
        return mock_data.WEB_ORDERS
    # Incremental pull: everything changed since the day before the logical run date.
    since = (run_date - timedelta(days=1)).isoformat()
    return paginate(session, os.environ["WEB_ORDERS_URL"], params={"updated_since": since},
                    headers=_auth("WEB_ORDERS_TOKEN"))


def mobile_orders(session, run_date, mock=False):
    if mock:
        return mock_data.MOBILE_ORDERS
    since = (run_date - timedelta(days=1)).isoformat()
    return paginate(session, os.environ["MOBILE_ORDERS_URL"], params={"updatedSince": since},
                    headers=_auth("MOBILE_ORDERS_TOKEN"), items_key="orders", next_key="nextPage")


def crm_tier_events(session, run_date, mock=False):
    if mock:
        return mock_data.CRM_TIER_EVENTS
    return paginate(session, os.environ["CRM_TIER_EVENTS_URL"], headers=_auth("CRM_TOKEN"))


def fx_rates(session, start, end, mock=False):
    """Daily USD->INR (and any other FX_BASES) rates from the Frankfurter API (ECB data).
    One record per base currency, kept exactly as the API returns it."""
    if mock:
        return mock_data.FX_RESPONSES
    base_url = os.getenv("FX_URL", "https://api.frankfurter.app")
    out = []
    for base in [b.strip() for b in os.getenv("FX_BASES", "USD").split(",") if b.strip()]:
        r = session.get(f"{base_url}/{start.isoformat()}..{end.isoformat()}",
                        params={"from": base, "to": "INR"}, timeout=30)
        r.raise_for_status()
        out.append(r.json())
    return out


def pincodes(session, pins, mock=False):
    """India Post lookup, one call per pincode. Only pincodes we don't know yet are asked."""
    base_url = os.getenv("PINCODE_URL", "https://api.postalpincode.in/pincode")
    out = []
    for pin in pins:
        if mock:
            resp = mock_data.pincode_response(pin)
        else:
            r = session.get(f"{base_url}/{pin}", timeout=30)
            r.raise_for_status()
            resp = r.json()
            time.sleep(0.2)                    # be polite to a free public API
        out.append({"pincode": pin, "response": resp})
    return out

In [ ]:
%%writefile /content/freshcart_elt/snowflake_io.py
"""LOAD: connect to Snowflake, upload landing files to an internal stage (PUT) and copy
them into the bronze tables (COPY INTO). Also runs the SQL transformation files."""
import logging
import os
from pathlib import Path

import snowflake.connector

log = logging.getLogger("snowflake")


def connect(bootstrap=False):
    """bootstrap=True connects without a database/warehouse (they may not exist yet
    before --setup has run; 00_setup.sql creates them and USEs them)."""
    kw = {
        "account": os.environ["SNOWFLAKE_ACCOUNT"],
        "user": os.environ["SNOWFLAKE_USER"],
        "role": os.getenv("SNOWFLAKE_ROLE"),
        "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
        "database": os.getenv("SNOWFLAKE_DATABASE"),
        "session_parameters": {"QUERY_TAG": "freshcart_elt"},
    }
    if bootstrap:
        kw.pop("warehouse"); kw.pop("database")
    if os.getenv("SNOWFLAKE_PRIVATE_KEY_FILE"):          # key-pair auth (recommended)
        kw["private_key_file"] = os.environ["SNOWFLAKE_PRIVATE_KEY_FILE"]
        kw["private_key_file_pwd"] = os.getenv("SNOWFLAKE_PRIVATE_KEY_FILE_PWD")
    elif os.getenv("SNOWFLAKE_AUTHENTICATOR"):           # e.g. externalbrowser (SSO)
        kw["authenticator"] = os.environ["SNOWFLAKE_AUTHENTICATOR"]
    else:
        kw["password"] = os.environ["SNOWFLAKE_PASSWORD"]
    return snowflake.connector.connect(**{k: v for k, v in kw.items() if v})


def run_sql_file(conn, path, variables=None):
    """Run every statement in a .sql file. Variables become session variables, so the SQL
    can use them as $run_id, $run_date."""
    text = Path(path).read_text(encoding="utf-8")
    prefix = "".join(f"SET {k} = '{v}';\n" for k, v in (variables or {}).items())
    for cur in conn.execute_string(prefix + text, remove_comments=True):
        cur.close()
    log.info("ran %s", Path(path).name)


def query_one(conn, sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return cur.fetchone()


def query_all(conn, sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return cur.fetchall()


def load_file(conn, local_path, table, stage_dir):
    """PUT one landing file to @bronze.landing/<stage_dir>/ then COPY it into <table>.

    COPY remembers which files it has loaded (for 64 days), so re-running this for the
    same file loads nothing the second time: the load step is idempotent."""
    if local_path is None:
        return 0
    uri = "file://" + Path(local_path).resolve().as_posix()
    with conn.cursor() as cur:
        cur.execute(f"PUT '{uri}' @bronze.landing/{stage_dir}/ AUTO_COMPRESS=FALSE OVERWRITE=FALSE")
        cur.execute(f"""
            COPY INTO {table} (source, raw, run_id, extracted_at, src_file, src_row)
            FROM (
              SELECT $1:source::STRING, $1:data, $1:run_id::STRING,
                     $1:extracted_at::TIMESTAMP_TZ, METADATA$FILENAME, METADATA$FILE_ROW_NUMBER
              FROM @bronze.landing/{stage_dir}/
            )
            FILE_FORMAT = (FORMAT_NAME = 'bronze.ff_ndjson')
            ON_ERROR = 'ABORT_STATEMENT'
        """)
        rows = cur.fetchall()
    # COPY returns one row per file: (file, status, rows_parsed, rows_loaded, ...),
    # or a single message row when there was nothing new to load.
    loaded = sum(int(r[3]) for r in rows if len(r) > 3 and str(r[1]).startswith(("LOADED", "PARTIALLY")))
    log.info("%s: %d rows copied into %s", Path(local_path).name, loaded, table)
    return loaded

In [ ]:
%%writefile /content/freshcart_elt/run_pipeline.py
"""FreshCart ELT pipeline.

    python run_pipeline.py --setup --mock    # first time: create objects, run on sample data
    python run_pipeline.py --mock            # run again on sample data (idempotent)
    python run_pipeline.py                   # real APIs from .env
    python run_pipeline.py --dry-run --mock  # extract only, no Snowflake needed

Order of work (each step needs the previous one):
  1 EXTRACT  APIs -> local landing files (raw JSON)
  2 LOAD     landing files -> Snowflake bronze tables (PUT + COPY INTO)
  3 TRANSFORM in Snowflake with SQL:
       10 cleanse orders -> 20 deduplicate -> (look up new pincodes) ->
       30 customers SCD2, 31 FX daily, 32 pincode geo -> 40 enrich + publish gold
  4 CHECK    data-quality assertions, then print the gold totals
"""
import argparse
import logging
import os
import sys
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

from dotenv import load_dotenv

import extract

SQL = Path(__file__).parent / "sql"
log = logging.getLogger("pipeline")


def parse_args():
    p = argparse.ArgumentParser(description="FreshCart ELT pipeline")
    p.add_argument("--setup", action="store_true", help="run sql/00_setup.sql first")
    p.add_argument("--mock", action="store_true", help="use built-in sample API responses")
    p.add_argument("--run-date", help="logical date YYYY-MM-DD (default: today)")
    p.add_argument("--dry-run", action="store_true", help="extract only; do not touch Snowflake")
    p.add_argument("--strict", action="store_true", help="fail (not warn) on quality warnings")
    return p.parse_args()


class Run:
    """Logs each step to the console and to ops.run_log, with timing."""
    def __init__(self, conn, run_id):
        self.conn, self.run_id = conn, run_id

    def record(self, step, status, detail=""):
        if self.conn is None:
            return
        with self.conn.cursor() as cur:
            cur.execute("INSERT INTO ops.run_log (run_id, step, status, detail) VALUES (%s, %s, %s, %s)",
                        (self.run_id, step, status, str(detail)[:4000]))

    def step(self, name, fn, *a, **kw):
        t0 = time.time()
        log.info("▶ %s", name)
        try:
            out = fn(*a, **kw)
        except Exception as e:
            self.record(name, "failed", repr(e))
            log.error("✖ %s failed: %s", name, e)
            raise
        secs = time.time() - t0
        self.record(name, "success", f"{secs:.1f}s; result={out!r}"[:4000])
        log.info("✔ %s (%.1fs)", name, secs)
        return out


def main():
    load_dotenv()
    args = parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)-9s %(message)s", datefmt="%H:%M:%S")

    run_date = date.fromisoformat(args.run_date) if args.run_date else date.today()
    run_id = f"{run_date:%Y%m%d}_{datetime.now(timezone.utc):%H%M%S}"
    mock = args.mock
    http = extract.http_session()
    fx_start = run_date - timedelta(days=int(os.getenv("FX_LOOKBACK_DAYS", "35")))
    log.info("run_id=%s run_date=%s mock=%s", run_id, run_date, mock)

    # ---------------- 1. EXTRACT (sources that don't depend on Snowflake) ----------------
    files = {
        "web":       extract.write_landing("web", extract.web_orders(http, run_date, mock), run_id, run_date),
        "mobile":    extract.write_landing("mobile", extract.mobile_orders(http, run_date, mock), run_id, run_date),
        "crm":       extract.write_landing("crm", extract.crm_tier_events(http, run_date, mock), run_id, run_date),
        "fx":        extract.write_landing("fx", extract.fx_rates(http, fx_start, run_date, mock), run_id, run_date),
    }
    if args.dry_run:
        log.info("dry run: landing files written, Snowflake not touched: %s", files)
        return

    import snowflake_io as sf                     # imported here so --dry-run needs no connector
    conn = sf.connect(bootstrap=args.setup)
    run = Run(None, run_id)
    try:
        if args.setup:
            run.step("setup", sf.run_sql_file, conn, SQL / "00_setup.sql")
        run.conn = conn                           # ops.run_log exists from here on
        vars_ = {"run_id": run_id, "run_date": run_date.isoformat()}

        # ---------------- 2. LOAD raw files into bronze ----------------
        stage = lambda src: f"{src}/{run_date.isoformat()}/{run_id}"
        run.step("load web orders",    sf.load_file, conn, files["web"],    "bronze.orders_raw",    stage("orders_web"))
        run.step("load mobile orders", sf.load_file, conn, files["mobile"], "bronze.orders_raw",    stage("orders_mobile"))
        run.step("load crm",           sf.load_file, conn, files["crm"],    "bronze.customers_raw", stage("crm"))
        run.step("load fx",            sf.load_file, conn, files["fx"],     "bronze.fx_raw",        stage("fx"))

        # ---------------- 3. TRANSFORM inside Snowflake ----------------
        run.step("10 cleanse orders",  sf.run_sql_file, conn, SQL / "10_silver_orders_clean.sql", vars_)
        run.step("20 deduplicate",     sf.run_sql_file, conn, SQL / "20_silver_orders_dedup.sql", vars_)

        # Pincodes depend on the orders: only look up valid ones we have never seen.
        pins = [r[0] for r in sf.query_all(conn, """
            SELECT DISTINCT o.pincode FROM silver.orders_dedup o
            WHERE NOT EXISTS (SELECT 1 FROM ref.pincode_geo g WHERE g.pincode = o.pincode)""")]
        log.info("new pincodes to look up: %s", pins)
        pin_file = extract.write_landing("pincode", extract.pincodes(http, pins, mock), run_id, run_date)
        run.step("load pincodes",      sf.load_file, conn, pin_file, "bronze.pincode_raw", stage("pincode"))

        run.step("30 customers scd2",  sf.run_sql_file, conn, SQL / "30_dim_customers_scd2.sql", vars_)
        run.step("31 fx daily",        sf.run_sql_file, conn, SQL / "31_ref_fx_daily.sql", vars_)
        run.step("32 pincode geo",     sf.run_sql_file, conn, SQL / "32_ref_pincode_geo.sql", vars_)
        run.step("40 enrich + gold",   sf.run_sql_file, conn, SQL / "40_gold_fct_orders.sql", vars_)

        # ---------------- 4. CHECK ----------------
        run.step("quality checks", quality_checks, conn, args.strict)
        n, revenue, points = sf.query_one(conn,
            "SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points) FROM gold.fct_orders")
        log.info("GOLD: %s orders, revenue ₹%s, %s loyalty points", n, revenue, points)
    finally:
        conn.close()


def quality_checks(conn, strict):
    """Assertions after the run. 'fail' stops the pipeline; 'warn' only logs,
    unless --strict is given."""
    import snowflake_io as sf
    max_q = float(os.getenv("QUARANTINE_MAX_RATE", "0.05"))
    checks = [
        ("no fan-out in enrichment (rows out - rows in)",
         "SELECT (SELECT COUNT(*) FROM silver.orders_enriched) - (SELECT COUNT(*) FROM silver.orders_dedup)",
         lambda v: v == 0, "fail"),
        ("one gold row per order_id (duplicates)",
         "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM gold.fct_orders", lambda v: v == 0, "fail"),
        ("orders with no customer match",
         "SELECT COUNT_IF(cust_id IS NULL) FROM silver.orders_enriched", lambda v: v == 0, "warn"),
        ("orders with an amount but no FX rate",
         "SELECT COUNT_IF(amount IS NOT NULL AND fx_rate IS NULL) FROM silver.orders_enriched",
         lambda v: v == 0, "warn"),
        ("orders with no city",
         "SELECT COUNT_IF(city IS NULL) FROM silver.orders_enriched", lambda v: v == 0, "warn"),
        (f"quarantine rate (max {max_q:.0%})",
         "SELECT COALESCE(COUNT_IF(ARRAY_SIZE(reasons) > 0) / NULLIF(COUNT(*), 0), 0) FROM silver.orders_checked",
         lambda v: float(v) <= max_q, "warn"),
    ]
    problems = []
    for name, sql, ok, severity in checks:
        value = sf.query_one(conn, sql)[0]
        good = ok(value)
        log.info("   %-48s %-8s %s", name, value, "OK" if good else severity.upper())
        if not good and (severity == "fail" or strict):
            problems.append(f"{name} = {value}")
    if problems:
        raise RuntimeError("quality checks failed: " + "; ".join(problems))
    return "passed"


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        logging.getLogger("pipeline").error("pipeline failed: %s", exc)
        sys.exit(1)

In [ ]:
%%writefile /content/freshcart_elt/sql/00_setup.sql
-- =====================================================================
-- One-time setup. Run with:  python run_pipeline.py --setup --mock
-- Needs a role that can create warehouses and databases (e.g. SYSADMIN).
-- Safe to re-run: everything is IF NOT EXISTS, except the two seed tables.
-- =====================================================================
CREATE WAREHOUSE IF NOT EXISTS FRESHCART_WH
  WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE FRESHCART_WH;

CREATE DATABASE IF NOT EXISTS FRESHCART;
USE DATABASE FRESHCART;

CREATE SCHEMA IF NOT EXISTS bronze;      -- raw API responses, unchanged
CREATE SCHEMA IF NOT EXISTS silver;      -- cleaned, typed, deduplicated
CREATE SCHEMA IF NOT EXISTS dim;         -- dimensions (customers)
CREATE SCHEMA IF NOT EXISTS ref;         -- reference data (fx, pincodes, tiers)
CREATE SCHEMA IF NOT EXISTS gold;        -- business-ready tables
CREATE SCHEMA IF NOT EXISTS quarantine;  -- rows that failed validation
CREATE SCHEMA IF NOT EXISTS ops;         -- pipeline run log

-- Landing area inside Snowflake for the files Python uploads
CREATE FILE FORMAT IF NOT EXISTS bronze.ff_ndjson TYPE = JSON COMPRESSION = AUTO;
CREATE STAGE IF NOT EXISTS bronze.landing FILE_FORMAT = bronze.ff_ndjson;

-- ---------- BRONZE: one table per source, raw JSON in a VARIANT column ----------
CREATE TABLE IF NOT EXISTS bronze.orders_raw (
  ingest_seq   NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER,   -- arrival order
  source       STRING,          -- 'web' or 'mobile'
  raw          VARIANT,         -- the API record exactly as received
  run_id       STRING,
  extracted_at TIMESTAMP_TZ,
  src_file     STRING,
  src_row      NUMBER,
  loaded_at    TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);
CREATE TABLE IF NOT EXISTS bronze.customers_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);
CREATE TABLE IF NOT EXISTS bronze.fx_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);
CREATE TABLE IF NOT EXISTS bronze.pincode_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

-- ---------- REF: small hand-maintained seed tables ----------
CREATE OR REPLACE TABLE ref.tier_multiplier AS
SELECT * FROM VALUES ('Silver', 1), ('Gold', 2), ('Platinum', 3) AS v(tier, multiplier);

CREATE OR REPLACE TABLE ref.state_region AS
SELECT * FROM VALUES
  ('Tamil Nadu','South'), ('Karnataka','South'), ('Kerala','South'), ('Andhra Pradesh','South'),
  ('Telangana','South'), ('Puducherry','South'),
  ('Maharashtra','West'), ('Gujarat','West'), ('Goa','West'), ('Rajasthan','West'),
  ('Delhi','North'), ('Haryana','North'), ('Punjab','North'), ('Uttar Pradesh','North'),
  ('Himachal Pradesh','North'), ('Jammu and Kashmir','North'), ('Uttarakhand','North'),
  ('West Bengal','East'), ('Odisha','East'), ('Bihar','East'), ('Jharkhand','East'), ('Assam','East'),
  ('Madhya Pradesh','Central'), ('Chhattisgarh','Central')
  AS v(state, region);

-- Filled incrementally from the India Post API (see 32_ref_pincode_geo.sql)
CREATE TABLE IF NOT EXISTS ref.pincode_geo (
  pincode STRING PRIMARY KEY, city STRING, state STRING, region STRING,
  updated_at TIMESTAMP_LTZ
);

-- ---------- QUARANTINE, GOLD, OPS ----------
CREATE TABLE IF NOT EXISTS quarantine.orders (
  run_id STRING, quarantined_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
  ingest_seq NUMBER, source STRING, order_id STRING, reasons ARRAY, row_data VARIANT
);

CREATE TABLE IF NOT EXISTS gold.fct_orders (
  order_id STRING PRIMARY KEY, order_date DATE, cust_id STRING, email STRING,
  tier STRING, multiplier NUMBER, amount NUMBER(12,2), currency STRING,
  fx_rate NUMBER(12,4), amount_inr NUMBER(14,2), loyalty_points NUMBER,
  is_high_value BOOLEAN, city STRING, state STRING, region STRING,
  updated_at TIMESTAMP_NTZ, row_hash NUMBER, published_at TIMESTAMP_LTZ
);

CREATE TABLE IF NOT EXISTS ops.run_log (
  run_id STRING, step STRING, status STRING, detail STRING,
  logged_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

In [ ]:
%%writefile /content/freshcart_elt/sql/10_silver_orders_clean.sql
-- =====================================================================
-- SILVER 1: unify the two order APIs, cleanse, validate, quarantine.
-- Input : bronze.orders_raw (raw JSON from 'web' and 'mobile')
-- Output: silver.orders_checked (all rows + reasons), silver.orders_clean (good rows),
--         quarantine.orders (bad rows, appended once per row)
-- =====================================================================
CREATE OR REPLACE TRANSIENT TABLE silver.orders_checked AS
WITH unified AS (
  -- The two APIs name and nest fields differently; map both to one set of columns.
  SELECT ingest_seq, source,
         raw:order_id::STRING   AS order_id,
         raw:cust_email::STRING AS cust_email,
         raw:cust_name::STRING  AS cust_name,
         raw:order_date::STRING AS order_date,
         raw:amount::STRING     AS amount,
         raw:currency::STRING   AS currency,
         raw:pincode::STRING    AS pincode,
         raw:updated_at::STRING AS updated_at
  FROM bronze.orders_raw WHERE source = 'web'
  UNION ALL
  SELECT ingest_seq, source,
         raw:orderId::STRING, raw:customer.email::STRING, raw:customer.name::STRING,
         raw:orderDate::STRING, raw:total.value::STRING, raw:total.currency::STRING,
         raw:deliveryPin::STRING, raw:updatedAt::STRING
  FROM bronze.orders_raw WHERE source = 'mobile'
),
typed AS (
  SELECT
    ingest_seq, source,
    TRIM(order_id)                                    AS order_id,
    LOWER(TRIM(cust_email))                           AS email,
    INITCAP(RTRIM(TRIM(cust_name), '.'))              AS name,
    COALESCE(
      TRY_TO_DATE(order_date, 'YYYY-MM-DD'),
      TRY_TO_DATE(order_date, 'DD/MM/YYYY'),          -- mobile API sends day first
      TRY_TO_DATE(order_date, 'MON DD, YYYY')
    )                                                 AS order_date,
    TRY_TO_DECIMAL(REGEXP_REPLACE(amount, '[^0-9.-]', ''), 12, 2) AS amount,
    UPPER(TRIM(currency))                             AS currency,
    TRIM(pincode)                                     AS pincode,
    TRY_TO_TIMESTAMP_NTZ(updated_at)                  AS updated_at
  FROM unified
)
SELECT *,
  ARRAY_CONSTRUCT_COMPACT(
    IFF(order_id IS NULL, 'missing_order_id', NULL),
    IFF(email IS NULL OR NOT REGEXP_LIKE(email, '^[^@\\s]+@[^@\\s]+\\.[a-z]{2,}$'), 'bad_email', NULL),
    IFF(pincode IS NULL OR NOT REGEXP_LIKE(pincode, '^[1-9][0-9]{5}$'), 'bad_pincode', NULL),
    IFF(order_date IS NULL, 'unparseable_date', NULL),
    IFF(updated_at IS NULL, 'unparseable_updated_at', NULL),
    IFF(amount <= 0, 'non_positive_amount', NULL)          -- NULL amount is allowed (flagged below)
  ) AS reasons
FROM typed;

-- Good rows
CREATE OR REPLACE TABLE silver.orders_clean AS
SELECT * EXCLUDE reasons, (amount IS NULL) AS amount_missing
FROM silver.orders_checked
WHERE ARRAY_SIZE(reasons) = 0;

-- Bad rows: kept with their reasons. The NOT EXISTS stops the same bad record being
-- quarantined again when a later run re-extracts it (same source, order, version).
INSERT INTO quarantine.orders (run_id, ingest_seq, source, order_id, reasons, row_data)
SELECT $run_id, c.ingest_seq, c.source, c.order_id, c.reasons, OBJECT_CONSTRUCT(*)
FROM silver.orders_checked c
WHERE ARRAY_SIZE(c.reasons) > 0
  AND NOT EXISTS (
        SELECT 1 FROM quarantine.orders q
        WHERE q.source = c.source
          AND EQUAL_NULL(q.order_id, c.order_id)
          AND EQUAL_NULL(q.row_data:updated_at::TIMESTAMP_NTZ, c.updated_at))
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.source, c.order_id, c.updated_at
                           ORDER BY c.ingest_seq DESC) = 1;

In [ ]:
%%writefile /content/freshcart_elt/sql/20_silver_orders_dedup.sql
-- =====================================================================
-- SILVER 2: one row per order_id, newest version wins.
-- Ties on updated_at are broken by arrival order, so reruns give the same answer.
-- =====================================================================
CREATE OR REPLACE TABLE silver.orders_dedup AS
SELECT *
FROM silver.orders_clean
QUALIFY ROW_NUMBER() OVER (
          PARTITION BY order_id
          ORDER BY updated_at DESC, ingest_seq DESC
        ) = 1;

In [ ]:
%%writefile /content/freshcart_elt/sql/30_dim_customers_scd2.sql
-- =====================================================================
-- DIM: build the SCD Type 2 customer table from the CRM's tier-change log.
-- Each change starts a period; the period ends the day before the next change.
-- =====================================================================
CREATE OR REPLACE TABLE dim.customers_scd2 AS
WITH events AS (
  SELECT raw:cust_id::STRING                 AS cust_id,
         LOWER(TRIM(raw:email::STRING))      AS email,
         INITCAP(TRIM(raw:tier::STRING))     AS tier,
         TRY_TO_DATE(raw:effective_from::STRING) AS valid_from,
         loaded_at, ingest_seq
  FROM bronze.customers_raw
  -- the same event may be extracted on many runs; keep the latest copy
  QUALIFY ROW_NUMBER() OVER (PARTITION BY cust_id, valid_from
                             ORDER BY loaded_at DESC, ingest_seq DESC) = 1
)
SELECT cust_id, email, tier, valid_from,
       COALESCE(DATEADD(day, -1, LEAD(valid_from) OVER (PARTITION BY cust_id ORDER BY valid_from)),
                '9999-12-31'::DATE)                                       AS valid_to,
       LEAD(valid_from) OVER (PARTITION BY cust_id ORDER BY valid_from) IS NULL AS is_current
FROM events
WHERE cust_id IS NOT NULL AND valid_from IS NOT NULL;

In [ ]:
%%writefile /content/freshcart_elt/sql/31_ref_fx_daily.sql
-- =====================================================================
-- REF: daily FX rates to INR, with no gaps.
-- 1. Flatten the API's {"rates": {"2026-09-01": {"INR": 83.2}, ...}} into rows.
-- 2. Rates are not published on weekends/holidays: fill each missing day with the
--    previous published rate, so every order date finds a rate.
-- 3. Add INR = 1.0 for every day, so rupee orders join like any other.
-- =====================================================================
CREATE OR REPLACE TABLE ref.fx_daily AS
WITH published AS (
  SELECT r.raw:base::STRING          AS currency,
         f.key::DATE                 AS rate_date,
         f.value:INR::NUMBER(12,4)   AS rate_to_inr
  FROM bronze.fx_raw r, LATERAL FLATTEN(input => r.raw:rates) f
  QUALIFY ROW_NUMBER() OVER (PARTITION BY currency, rate_date
                             ORDER BY r.loaded_at DESC, r.ingest_seq DESC) = 1
),
bounds AS (
  SELECT MIN(rate_date) AS d0, GREATEST(MAX(rate_date), CURRENT_DATE()) AS d1 FROM published
),
spine AS (                                   -- one row per calendar day d0..d1
  SELECT DATEADD(day, ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, b.d0) AS d
  FROM bounds b, TABLE(GENERATOR(ROWCOUNT => 4000))
  QUALIFY d <= (SELECT d1 FROM bounds)
),
filled AS (
  SELECT c.currency, s.d AS rate_date,
         LAST_VALUE(p.rate_to_inr) IGNORE NULLS OVER (
           PARTITION BY c.currency ORDER BY s.d
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rate_to_inr,
         p.rate_to_inr IS NULL AS is_carried_forward
  FROM spine s
  CROSS JOIN (SELECT DISTINCT currency FROM published) c
  LEFT JOIN published p ON p.currency = c.currency AND p.rate_date = s.d
)
SELECT currency, rate_date, rate_to_inr, is_carried_forward FROM filled
UNION ALL
SELECT 'INR', d, 1.0, FALSE FROM spine;

In [ ]:
%%writefile /content/freshcart_elt/sql/32_ref_pincode_geo.sql
-- =====================================================================
-- REF: add newly looked-up pincodes to ref.pincode_geo.
-- India Post gives district and state; the company's sales region comes from
-- the ref.state_region seed table.
-- =====================================================================
MERGE INTO ref.pincode_geo t
USING (
  SELECT p.pincode, p.city, p.state, sr.region
  FROM (
    SELECT raw:pincode::STRING                                  AS pincode,
           raw:response[0]:PostOffice[0]:District::STRING       AS city,
           raw:response[0]:PostOffice[0]:State::STRING          AS state,
           loaded_at, ingest_seq
    FROM bronze.pincode_raw
    WHERE raw:response[0]:Status::STRING = 'Success'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY pincode ORDER BY loaded_at DESC, ingest_seq DESC) = 1
  ) p
  LEFT JOIN ref.state_region sr ON sr.state = p.state
) s
ON t.pincode = s.pincode
WHEN MATCHED AND (t.city <> s.city OR t.state <> s.state OR NVL(t.region,'') <> NVL(s.region,''))
  THEN UPDATE SET city = s.city, state = s.state, region = s.region, updated_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED
  THEN INSERT (pincode, city, state, region, updated_at)
       VALUES (s.pincode, s.city, s.state, s.region, CURRENT_TIMESTAMP());

In [ ]:
%%writefile /content/freshcart_elt/sql/40_gold_fct_orders.sql
-- =====================================================================
-- ENRICH + PUBLISH
-- 1. silver.orders_enriched: point-in-time joins to customer tier and FX rate,
--    computed columns, geography. LEFT JOINs so no order is silently lost.
-- 2. gold.fct_orders: idempotent MERGE. Re-running changes nothing unless a
--    value actually changed (row_hash).
-- =====================================================================
CREATE OR REPLACE TABLE silver.orders_enriched AS
SELECT o.*,
       c.cust_id, c.tier,
       m.multiplier,
       fx.rate_to_inr                              AS fx_rate,
       ROUND(o.amount * fx.rate_to_inr, 2)         AS amount_inr,
       FLOOR(amount_inr / 100) * m.multiplier      AS loyalty_points,
       amount_inr >= 2000                          AS is_high_value,
       g.city, g.state, g.region
FROM silver.orders_dedup o
LEFT JOIN dim.customers_scd2 c
       ON  c.email = o.email
       AND o.order_date BETWEEN c.valid_from AND c.valid_to     -- tier ON the order date
LEFT JOIN ref.tier_multiplier m ON m.tier = c.tier
LEFT JOIN ref.fx_daily fx
       ON  fx.currency = o.currency AND fx.rate_date = o.order_date  -- rate OF the order date
LEFT JOIN ref.pincode_geo g ON g.pincode = o.pincode;

MERGE INTO gold.fct_orders t
USING (
  SELECT order_id, order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate,
         amount_inr, loyalty_points, is_high_value, city, state, region, updated_at,
         HASH(order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate,
              amount_inr, loyalty_points, is_high_value, city, state, region, updated_at) AS row_hash
  FROM silver.orders_enriched
) s
ON t.order_id = s.order_id
WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET
  order_date = s.order_date, cust_id = s.cust_id, email = s.email, tier = s.tier,
  multiplier = s.multiplier, amount = s.amount, currency = s.currency, fx_rate = s.fx_rate,
  amount_inr = s.amount_inr, loyalty_points = s.loyalty_points, is_high_value = s.is_high_value,
  city = s.city, state = s.state, region = s.region, updated_at = s.updated_at,
  row_hash = s.row_hash, published_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT
  (order_id, order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate, amount_inr,
   loyalty_points, is_high_value, city, state, region, updated_at, row_hash, published_at)
VALUES
  (s.order_id, s.order_date, s.cust_id, s.email, s.tier, s.multiplier, s.amount, s.currency,
   s.fx_rate, s.amount_inr, s.loyalty_points, s.is_high_value, s.city, s.state, s.region,
   s.updated_at, s.row_hash, CURRENT_TIMESTAMP());

## Security note before running
> The uploaded source notebook contained a Snowflake password directly in a code cell. I removed it from this revised copy. **Rotate/change that password in Snowflake before continuing**, because it was shared in an uploaded notebook. Store the replacement only in Colab Secrets.

Add these keys in Colab's Secrets panel (🔑): `SNOWFLAKE_ACCOUNT`, `SNOWFLAKE_USER`, `SNOWFLAKE_PASSWORD`, and optionally `SNOWFLAKE_ROLE`, `SNOWFLAKE_WAREHOUSE`, `SNOWFLAKE_DATABASE`. Enable Notebook access for each.


## 3. Read your Snowflake login from Colab Secrets
The values go into environment variables, which the pipeline script reads. Nothing is printed.

In [ ]:
import os
from google.colab import userdata

def secret(name, required=False):
    try: value = userdata.get(name)
    except Exception: value = None
    if required and not value:
        raise RuntimeError(f"Add {name} in Colab Secrets and enable Notebook access.")
    return value

# Credentials are read only from Colab Secrets; never hard-code them in this notebook.
os.environ["SNOWFLAKE_ACCOUNT"] = secret("SNOWFLAKE_ACCOUNT", required=True)
os.environ["SNOWFLAKE_USER"] = secret("SNOWFLAKE_USER", required=True)
os.environ["SNOWFLAKE_PASSWORD"] = secret("SNOWFLAKE_PASSWORD", required=True)
os.environ["SNOWFLAKE_ROLE"] = secret("SNOWFLAKE_ROLE") or "SYSADMIN"
os.environ["SNOWFLAKE_WAREHOUSE"] = secret("SNOWFLAKE_WAREHOUSE") or "FRESHCART_WH"
os.environ["SNOWFLAKE_DATABASE"] = secret("SNOWFLAKE_DATABASE") or "FRESHCART"
os.environ["LANDING_DIR"] = "/content/freshcart_elt/landing"
os.environ["QUARANTINE_MAX_RATE"] = "0.25"
for key in ["SNOWFLAKE_PRIVATE_KEY_FILE", "SNOWFLAKE_PRIVATE_KEY_FILE_PWD", "SNOWFLAKE_AUTHENTICATOR"]:
    os.environ.pop(key, None)
print("Snowflake credentials loaded from Colab Secrets for user:", os.environ["SNOWFLAKE_USER"])


## 4. Test the connection

In [ ]:
import sys, os, importlib

# Make the project folder importable, and reload the module in case an older copy is cached
sys.path.insert(0, "/content/freshcart_elt")
if not os.path.exists("/content/freshcart_elt/snowflake_io.py"):
    raise SystemExit("snowflake_io.py not found: run the step 2 cells (the %%writefile ones) first.")
import snowflake_io as sf
importlib.reload(sf)

print("Connecting to", os.environ.get("SNOWFLAKE_ACCOUNT"), "as", os.environ.get("SNOWFLAKE_USER"), "...")

try:
    conn = sf.connect(bootstrap=True)
    account, user, role, version = sf.query_one(
        conn, "SELECT CURRENT_ACCOUNT(), CURRENT_USER(), CURRENT_ROLE(), CURRENT_VERSION()")
    print(f" Connected!  account={account}  user={user}  role={role}  version={version}")
    conn.close()

except KeyError as e:
    print(f"Setting {e} is missing: run the login cell (step 3) first.")

except Exception as e:
    msg = str(e)
    print(" Connection failed:\n ", msg[:500])
    low = msg.lower()
    if "incorrect username or password" in low:
        print("→ Check the user name and password in step 3.")
    elif "mfa" in low or "multi-factor" in low:
        print("→ Your account needs MFA. In Snowflake: your name → Settings → Authentication →")
        print("  Programmatic access tokens → Generate. Put the token in SNOWFLAKE_PASSWORD in step 3.")
    elif "404" in low or "could not connect" in low or "account" in low or "certificate" in low:
        print("→ Account identifier is probably wrong. In Snowflake: your name (bottom left) →")
        print("  your account → 'Copy account identifier' (looks like ABCDEFG-XY12345),")
        print("  put it in SNOWFLAKE_ACCOUNT in step 3, re-run step 3, then this cell.")

## 5. Dry run: extract only, no Snowflake
Writes the raw sample API responses to `landing/`. Then we peek at one raw mobile record.

In [ ]:
%cd /content/freshcart_elt
!python run_pipeline.py --dry-run --mock --run-date 2026-09-30
!zcat landing/mobile/*/*.json.gz | head -1

## 6. Run the full pipeline on the sample data
`--setup` creates the warehouse, database, schemas and tables. It is only needed the first time; leaving it in is harmless.

Expected last line: **GOLD: 4 orders, revenue ₹7785.00, 126 loyalty points**. One `WARN` for the quarantine rate (2 of 8 rows = 25%) is expected for this sample.

In [ ]:
%cd /content/freshcart_elt
!python run_pipeline.py --setup --mock --run-date 2026-09-30

## 7. Look inside each layer
`show()` runs a query and returns a table.

In [ ]:
import pandas as pd
conn = sf.connect()

def show(sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        return pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])

show("SELECT ingest_seq, source, raw FROM bronze.orders_raw ORDER BY ingest_seq")

### Silver: cleaned rows (6)

In [ ]:
show("SELECT ingest_seq, source, order_id, email, name, order_date, amount, currency, pincode, amount_missing FROM silver.orders_clean ORDER BY ingest_seq")

### Quarantine: rejected rows and why

In [ ]:
show("SELECT run_id, source, order_id, reasons FROM quarantine.orders")

### Silver: one row per order (4)

In [ ]:
show("SELECT order_id, ingest_seq, updated_at, amount FROM silver.orders_dedup ORDER BY order_id")

### Dim: customer tier history (SCD2)

In [ ]:
show("SELECT * FROM dim.customers_scd2 ORDER BY cust_id, valid_from")

### Ref: daily FX rates, weekends filled

In [ ]:
show("SELECT * FROM ref.fx_daily WHERE currency = 'USD' ORDER BY rate_date LIMIT 10")

### Ref: pincode geography

In [ ]:
show("SELECT * FROM ref.pincode_geo")

### Gold: the final table

In [ ]:
show("SELECT order_id, order_date, tier, amount, currency, fx_rate, amount_inr, loyalty_points, is_high_value, city, region FROM gold.fct_orders ORDER BY order_id")

### Gold totals

In [ ]:
show("SELECT COUNT(*) AS orders, SUM(amount_inr) AS revenue_inr, SUM(loyalty_points) AS points FROM gold.fct_orders")

### Run log

In [ ]:
show("SELECT step, status, detail, logged_at FROM ops.run_log ORDER BY logged_at DESC LIMIT 20")

## 8. Run it again: nothing should double
A second run extracts the same sample again, so bronze gets new copies. Deduplication and the gold `MERGE` absorb them: the totals stay 4 orders, ₹7,785.00, 126 points.

In [ ]:
!python run_pipeline.py --mock --run-date 2026-09-30
show("SELECT COUNT(*) AS bronze_rows FROM bronze.orders_raw")

In [ ]:
show("SELECT COUNT(*) AS orders, SUM(amount_inr) AS revenue_inr, SUM(loyalty_points) AS points FROM gold.fct_orders")

## 9. Switch to your real APIs
Add these to Colab Secrets, run the cell, then run the pipeline **without** `--mock`.
Exchange rates (frankfurter.app) and pincodes (api.postalpincode.in) are free public APIs and need no key.

If your APIs page their results differently, edit `items_key` / `next_key` in `extract.py`. If their field names differ, edit only the `unified` block at the top of `sql/10_silver_orders_clean.sql`.

In [ ]:
for k in ["WEB_ORDERS_URL", "WEB_ORDERS_TOKEN", "MOBILE_ORDERS_URL", "MOBILE_ORDERS_TOKEN",
          "CRM_TIER_EVENTS_URL", "CRM_TOKEN"]:
    v = secret(k)
    if v:
        os.environ[k] = v
os.environ["FX_URL"] = "https://api.frankfurter.app"
os.environ["FX_BASES"] = "USD"
os.environ["PINCODE_URL"] = "https://api.postalpincode.in/pincode"

# !python run_pipeline.py          # uncomment to run on today's real data

## 10. Clean up (optional)
Removes everything this notebook created in Snowflake. Uncomment to run.

In [ ]:
# conn.cursor().execute("DROP DATABASE IF EXISTS FRESHCART")
# conn.cursor().execute("DROP WAREHOUSE IF EXISTS FRESHCART_WH")
conn.close()